# 03. RabbitMQ Deep Dive - 6가지 메시징 패턴 완전 정복

> **난이도**: 중급 | **예상 소요**: 35분 | **사전 지식**: [01. 프로젝트 개요](./01_project_overview.ipynb) 완료

---

## [목차]
1. [환경 설정](#1.-환경-설정)
2. [AMQP 0-9-1 프로토콜 개요](#2.-AMQP-0-9-1-프로토콜-개요)
3. [connect_robust와 prefetch_count](#3.-connect_robust와-prefetch_count-분석)
4. [Pattern 1: Direct Queue](#4.-Pattern-1:-Direct-Queue) -- Point-to-Point
5. [Pattern 2: Fanout Exchange](#5.-Pattern-2:-Fanout-Exchange) -- Broadcast
6. [Pattern 3: Topic Exchange](#6.-Pattern-3:-Topic-Exchange) -- 라우팅 키 패턴
7. [Pattern 4: DLQ](#7.-Pattern-4:-DLQ) -- Dead Letter Queue
8. [Pattern 5: Priority Queue](#8.-Pattern-5:-Priority-Queue) -- 우선순위
9. [Pattern 6: TTL](#9.-Pattern-6:-TTL) -- 메시지 만료
10. [Management UI 활용법](#10.-RabbitMQ-Management-UI-활용법)
11. [정리 및 요약](#11.-정리-및-요약)

---

## [학습 목표]
- AMQP 0-9-1의 Exchange -> Binding -> Queue 모델 이해
- 6가지 메시징 패턴을 실제 API 호출로 실습
- DLQ로 실패 메시지를 격리하는 방법 체험

> **Tip**: RabbitMQ Management UI(`http://localhost:15672`)를 함께 열어두면 큐의 상태를 시각적으로 확인할 수 있습니다.

> **관련 노트북**: Headers Exchange, Retry+Exponential Backoff는 [07. 고급 패턴](./07_advanced_patterns.ipynb)에서 다룹니다.

---

### [노트북 네비게이션]
| 이전 | 현재 | 다음 |
|------|------|------|
| [02. Redis Deep Dive](./02_redis_deep_dive.ipynb) | **03. RabbitMQ Deep Dive** (현재) | [04. Kafka Deep Dive](./04_kafka_deep_dive.ipynb) |

| # | 패턴 | 핵심 개념 |
|---|------|----------|
| 1 | Direct Queue | Point-to-Point, Default Exchange |
| 2 | Fanout Exchange | Broadcast, 모든 바인딩 큐에 전달 |
| 3 | Topic Exchange | Routing Key 패턴 매칭 (`*`, `#`) |
| 4 | DLQ (Dead Letter Queue) | 실패 메시지 격리, DLX |
| 5 | Priority Queue | 우선순위 기반 소비 순서 |
| 6 | TTL (Time-To-Live) | 메시지 자동 만료 |

## 1. 환경 설정

프로젝트 루트를 `sys.path`에 추가하고, `httpx.AsyncClient`로 FastAPI 엔드포인트를 호출할 준비를 합니다.
모든 API 호출은 비동기(`await`)로 수행합니다.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = str(Path.cwd().parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import httpx
BASE_URL = "http://localhost:8000"
client = httpx.AsyncClient(base_url=BASE_URL, timeout=10.0)

서버 헬스체크를 통해 연결 상태를 확인합니다. 응답이 오지 않으면 서버가 기동되지 않은 것입니다.

In [2]:
resp = await client.get("/health")
print(f"Status: {resp.status_code}")
resp.json()

Status: 200


{'api': 'running',
 'redis': 'connected',
 'rabbitmq': 'connected',
 'kafka': 'connected'}

## 2. AMQP 0-9-1 프로토콜 개요

### 왜 AMQP에는 Exchange라는 것이 있을까?

메시지를 보내는 방법을 생각해봅시다. 가장 단순한 방법은 **보내는 사람이 받는 사람에게 직접 전달**하는 것입니다.
Redis Pub/Sub가 바로 이 방식입니다:

```text
[Redis Pub/Sub - 직접 전달 방식]

  Publisher ---publish "news"--> [channel: news] ---> Subscriber A
                                                 ---> Subscriber B
  
  * Publisher가 "news" 채널에 직접 메시지를 보냄
  * 단순하지만, "뉴스 중 스포츠만 받고 싶다"는 요구를 처리하기 어려움
```

하지만 현실에서는 이런 요구가 생깁니다:
- **하나의 메시지를 여러 곳에 보내고 싶다** (예: 주문이 생기면 재고팀 + 배송팀 + 알림팀 모두에게)
- **조건에 따라 다른 곳에 보내고 싶다** (예: 한국 주문은 한국팀에, 미국 주문은 미국팀에)

이런 유연한 라우팅을 하려면, **보내는 사람과 받는 사람 사이에 "분류기"가 필요합니다.**
RabbitMQ는 이 분류기를 **Exchange**라고 부릅니다.

---

### 우체국 시스템으로 이해하는 AMQP

RabbitMQ의 구조를 **우체국 시스템**에 비유하면 직관적으로 이해할 수 있습니다:

| AMQP 구성 요소 | 우체국 비유 | 역할 |
|---------------|-----------|------|
| **Producer** | 편지를 보내는 사람 | 메시지를 만들어서 우체국(Exchange)에 맡김 |
| **Exchange** | 우체국 분류기 | 편지를 어디로 보낼지 결정하는 중앙 장치 |
| **Binding** | 배달 규칙표 | "이런 편지는 이 우체통에 넣어라"라는 규칙 |
| **Queue** | 우체통 | 편지가 실제로 쌓이는 곳 |
| **Consumer** | 수신자 | 우체통에서 편지를 꺼내 읽는 사람 |

```text
[우체국 시스템 비유]

  편지를 보내는 사람        우체국 분류기         배달 규칙          우체통            수신자
  (Producer)              (Exchange)          (Binding)         (Queue)          (Consumer)

  +----------+        +----------------+                    +-----------+
  | 김철수가  | -----> |                | -- "서울행" ------> | 서울 우체통 | ----> 서울 수신자
  | 편지 투입 |        |   우체국에서    |                    +-----------+
  +----------+        |   분류합니다    |                    +-----------+
                      |                | -- "부산행" ------> | 부산 우체통 | ----> 부산 수신자
                      +----------------+                    +-----------+

  * 김철수는 편지에 "서울"이라고 적어서 우체국에 맡깁니다 (= routing_key 지정)
  * 우체국 분류기는 규칙에 따라 서울 우체통에 넣습니다 (= Exchange가 Binding 규칙으로 라우팅)
  * 서울 수신자가 우체통에서 편지를 꺼냅니다 (= Consumer가 Queue에서 메시지 소비)
```

---

### Redis와 비교: 왜 RabbitMQ는 Exchange가 필요한가?

```text
[Redis Pub/Sub]                          [RabbitMQ AMQP]

Publisher --> Channel --> Subscriber     Producer --> Exchange --> Binding --> Queue --> Consumer
                                                      |
              (중간 분류 없음)                          (분류기가 있어서 유연한 라우팅 가능!)
                                                      |
                                                      +-- direct: 정확한 이름으로 1:1 전달
                                                      +-- fanout: 모든 큐에 복사 전달
                                                      +-- topic:  패턴 매칭으로 선택 전달
                                                      +-- headers: 헤더 값으로 분류
```

**핵심 차이**: Redis는 채널에 직접 발행하지만, RabbitMQ는 Exchange라는 분류기를 거칩니다.
이 덕분에 **같은 메시지를 여러 곳에 보내거나, 조건에 따라 다른 곳에 보내는 것**이 자연스럽게 가능합니다.

### Exchange 타입 요약

| Exchange 타입 | 우체국 비유 | 라우팅 방식 |
|-------------|-----------|-----------|
| **direct** | "이 주소로 정확히 배달해" | routing_key가 정확히 일치하는 큐에 전달 |
| **fanout** | "모든 우체통에 복사해서 넣어" | 바인딩된 모든 큐에 무조건 전달 |
| **topic** | "서울로 시작하는 주소면 서울 우체통에" | routing_key 패턴(`*`, `#`)으로 매칭 |
| **headers** | "편지 봉투의 속성을 보고 분류해" | 메시지 헤더 값으로 매칭 |

> **Default Exchange**는 이름이 `""`인 특수한 direct exchange입니다.
> 모든 큐가 자동으로 "큐 이름 = routing_key"로 바인딩되어 있어서, 별도 설정 없이 큐 이름만으로 메시지를 보낼 수 있습니다.

## 3. `connect_robust`와 `prefetch_count` 분석

우리 프로젝트의 `RabbitMQBroker.connect()` 메서드를 살펴봅니다.

```python
# app/brokers/rabbitmq_broker.py
async def connect(self) -> None:
    self.connection = await aio_pika.connect_robust(settings.rabbitmq_url)
    self.channel = await self.connection.channel()
    await self.channel.set_qos(prefetch_count=10)
```

### `connect_robust` vs `connect`

| 항목 | `connect` | `connect_robust` |
|------|-----------|-------------------|
| 자동 재연결 | X | O |
| Exchange/Queue 재선언 | X | O |
| 프로덕션 권장 | X | **O** |

### `prefetch_count=10`의 의미

Consumer가 한 번에 최대 **10개**의 unacked 메시지를 받을 수 있습니다.  
이 값이 너무 크면 특정 Consumer에 메시지가 몰리고, 너무 작으면 처리량이 떨어집니다.

```text
RabbitMQ Queue:  [msg1][msg2][msg3][msg4]...[msg10][msg11][msg12]...
                  \________________________/        ↑
                   Consumer에게 전달됨 (10개)       대기 중 (prefetch 한도)
```


### Competing Consumers 패턴 (경쟁적 소비)

`prefetch_count`는 **Competing Consumers** 패턴의 핵심입니다.
여러 Consumer가 같은 큐에서 메시지를 경쟁적으로 가져가는 구조입니다.

```text
                              ┌─── Consumer A (prefetch=1, 느린 워커)
Queue: [1][2][3][4][5]... ────┼─── Consumer B (prefetch=1, 빠른 워커)  
                              └─── Consumer C (prefetch=1, 보통 워커)
```

**prefetch_count=1이면**: 한 번에 메시지 1개만 받고, ACK 후 다음 메시지를 받음
→ 느린 워커에게 메시지가 몰리지 않아 **공정한 분배**가 됩니다.

**prefetch_count=10이면**: 한 번에 10개를 미리 가져감
→ 빠른 워커에게 유리하지만, 워커 장애 시 10개가 재처리 대상이 됩니다.

| prefetch_count | 처리량 | 공정성 | 장애 시 영향 |
|----------------|--------|--------|-------------|
| 1 | 낮음 | 높음 (fair dispatch) | 최소 (1건만 재처리) |
| 10 (프로젝트 설정) | 높음 | 보통 | 중간 (10건 재처리) |
| 무제한 | 최대 | 낮음 (한 워커에 집중) | 최대 |

이 프로젝트는 `prefetch_count=10`으로 **처리량과 공정성의 균형**을 잡고 있습니다.
CPU-intensive 작업이라면 1, I/O-bound 작업이라면 10~50이 일반적입니다.

---
## 4. Pattern 1: Direct Queue (Point-to-Point)

### 택배 직접 배송으로 이해하기

Direct Queue는 **"이름으로 직접 찾아가는 택배"**와 같습니다.

보내는 사람이 택배 상자에 **받는 사람 이름**(= 큐 이름)을 적으면,
택배 기사(Default Exchange)가 **그 이름의 집(Queue)에 직접 배달**합니다.
별도의 분류 규칙(Binding)을 설정할 필요가 없습니다.

```text
[택배 직접 배송 비유]

  보내는 사람              택배 기사                  받는 사람 집
  (Producer)         (Default Exchange "")         (Queue)

  +-------------+     +------------------+     +----------------+
  | 상자에 적음: |     |  "order-queue"   |     |  order-queue   |
  | 받는 곳:     | --> |  이름을 보고     | --> |  집 앞에 상자   | --> 수신자가
  | "order-queue"|     |  직접 배달!      |     |  쌓임          |    꺼내감
  +-------------+     +------------------+     +----------------+
                       routing_key="order-queue"

  * Default Exchange는 모든 큐가 "큐이름 = routing_key"로 자동 바인딩되어 있음
  * 그래서 별도 Exchange 선언이나 Binding 설정 없이 바로 사용 가능
```

### 왜 durable과 PERSISTENT가 필요한가?

택배에도 **일반 택배**와 **등기 택배**가 있듯이, RabbitMQ 메시지에도 안전 등급이 있습니다:

| 설정 | 비유 | 의미 |
|------|------|------|
| `durable=True` | 영구 사서함 (우체국이 문을 닫았다 열어도 사서함은 그대로) | 브로커가 재시작되어도 큐가 사라지지 않음 |
| `durable=False` | 임시 사서함 (우체국 문 닫으면 사서함도 사라짐) | 브로커 재시작 시 큐가 삭제됨 |
| `DeliveryMode.PERSISTENT` | 등기 우편 (배달 기록을 디스크에 남김) | 메시지가 디스크에 기록되어 브로커 재시작 후에도 유지 |
| `DeliveryMode.NOT_PERSISTENT` | 일반 우편 (메모리에만 보관) | 브로커 재시작 시 메시지 유실 가능 |

> **주의**: `durable=True`(큐 유지) + `PERSISTENT`(메시지 유지) **둘 다** 설정해야 완전한 메시지 보존이 됩니다.
> 큐는 살아 있는데 안의 메시지가 사라지거나, 메시지는 저장했는데 큐 자체가 없어지면 의미가 없기 때문입니다.

### 언제 Direct Queue를 사용하나?

- **1:1 작업 분배**: 주문 처리, 결제 처리처럼 **하나의 메시지를 정확히 하나의 Consumer가 처리**해야 할 때
- 라우팅 규칙이 필요 없고, **큐 이름만으로 충분**할 때
- 가장 단순한 패턴이므로, 복잡한 라우팅이 필요 없다면 이것부터 고려하세요

Direct Queue에 메시지를 발행합니다. `POST /rabbitmq/direct/publish` 엔드포인트를 사용합니다.

In [6]:
resp = await client.post(
    "/rabbitmq/direct/publish",
    params={"queue_name": "order-queue"},
    json={"content": "create_order", "metadata": {"item": "laptop", "qty": 2}}
)
print(f"Status: {resp.status_code}")
resp.json()

Status: 200


{'broker': 'rabbitmq',
 'pattern': 'direct',
 'destination': 'test-queue',
 'queue_message_count': 4,
 'elapsed_ms': 21.532}

큐 상태를 조회하여 메시지가 적재되었는지 확인합니다.

In [7]:
resp = await client.get("/rabbitmq/queue/info/order-queue")
print(f"Queue Info: {resp.json()}")

Queue Info: {'queue': 'order-queue', 'error': "NOT_FOUND - no queue 'order-queue' in vhost '/'"}


### Direct Queue 결과 분석

응답에서 확인할 포인트:
- `pattern: "direct"` - Default Exchange를 통한 직접 전달
- `queue_message_count` - 발행 직전의 큐 메시지 수 (발행 후 1 증가 예상)
- Consumer가 없으면 메시지는 큐에 계속 쌓입니다

---
## 5. Pattern 2: Fanout Exchange (Broadcast)

### 교내 방송으로 이해하기

Fanout Exchange는 **학교 교내 방송**과 같습니다.

교장 선생님(Producer)이 방송실(Fanout Exchange)에서 "오늘 체육대회입니다"라고 방송하면,
**모든 교실(Queue)에 동시에 같은 내용이 전달**됩니다.
어떤 교실에 보낼지 선택하는 것이 아니라, **연결된 모든 곳에 복사**됩니다.

```text
[교내 방송 비유]

  교장 선생님              방송 시스템              각 교실               학생들
  (Producer)          (Fanout Exchange)          (Queue)             (Consumer)

                                              +---------------+
                                         +--> | 1학년 1반 교실 | --> 1-1반 학생들
  +------------+     +--------------+    |    +---------------+
  | "오늘은    |     |              |    |
  |  체육      | --> | 모든 교실에   | ---+    +---------------+
  |  대회!"    |     | 동시 방송!   |    +--> | 2학년 3반 교실 | --> 2-3반 학생들
  +------------+     +--------------+    |    +---------------+
                      routing_key 무시   |
                                         |    +---------------+
                                         +--> | 3학년 2반 교실 | --> 3-2반 학생들
                                              +---------------+

  * 방송은 routing_key를 무시합니다 -- 어떤 값을 넣어도 모든 곳에 전달
  * 방송 스피커가 설치된 교실(바인딩된 큐)에만 전달됨
  * 스피커가 없는 교실(바인딩 안 된 큐)에는 전달되지 않음
```

### Direct와 Fanout의 차이

```text
[Direct - 1:1 택배 배달]              [Fanout - 1:N 교내 방송]

  Producer --> Queue A (만 받음)       Producer --> Queue A (받음)
                                                --> Queue B (받음)
                                                --> Queue C (받음)

  "이 택배는 A한테만 가"               "이 방송은 모두가 듣는다"
```

### Redis Pub/Sub와의 비교

Redis Pub/Sub도 1:N 브로드캐스트를 지원하지만, RabbitMQ Fanout과는 중요한 차이가 있습니다:

| 비교 항목 | Redis Pub/Sub | RabbitMQ Fanout |
|----------|--------------|-----------------|
| 메시지 저장 | 저장하지 않음 (발행 시점에 연결된 구독자만 받음) | **큐에 저장됨** (Consumer가 나중에 와도 받을 수 있음) |
| Consumer 오프라인 | 메시지 유실 | 큐에 쌓여서 대기 |
| 비유 | 라디오 방송 (안 틀면 못 들음) | 녹화 방송 (나중에 볼 수 있음) |

> **핵심**: RabbitMQ Fanout은 메시지가 큐에 저장되므로, Consumer가 잠시 꺼져 있다가 다시 연결해도 밀린 메시지를 모두 받을 수 있습니다.

**사용 사례:** 이벤트 알림 (회원 가입 시 -> 이메일팀 + 포인트팀 + 분석팀 동시 통보), 캐시 무효화, 로그 팬아웃

먼저 Fanout Exchange에 큐를 바인딩합니다. 바인딩 없이 발행하면 메시지는 버려집니다.

In [ ]:
# 2개의 큐를 fanout exchange에 바인딩
for q in ["notifications", "audit-log"]:
    resp = await client.post(
        "/rabbitmq/fanout/bind",
        params={"queue_name": q}
    )
    print(f"Bind {q}: {resp.json()}")

Fanout Exchange에 메시지를 발행합니다. 바인딩된 모든 큐(`notifications`, `audit-log`)에 동일한 메시지가 전달됩니다.

In [ ]:
resp = await client.post(
    "/rabbitmq/fanout/publish",
    json={"content": "user_signup", "metadata": {"user_id": 42}}
)
print(f"Fanout publish: {resp.json()}")

각 큐의 상태를 조회하여 두 큐 모두에 메시지가 도착했는지 확인합니다.

In [ ]:
for q in ["notifications", "audit-log"]:
    resp = await client.get(f"/rabbitmq/queue/info/{q}")
    print(f"{q}: {resp.json()}")

---
## 6. Pattern 3: Topic Exchange (Routing Key 패턴 매칭)

### 신문 구독으로 이해하기

Topic Exchange는 **신문 구독 서비스**와 같습니다.

신문사(Producer)는 기사를 **카테고리 태그**(routing_key)를 붙여서 발행합니다.
예: `sports.soccer.korea`, `sports.baseball.japan`, `economy.stock.korea`

구독자(Consumer)는 자신이 관심 있는 **카테고리 패턴**(binding_key)을 등록합니다.
신문사의 분류 시스템(Topic Exchange)이 패턴에 맞는 구독자에게만 기사를 전달합니다.

```text
[신문 구독 비유]

  신문사가 기사 발행                   분류 시스템               구독자 사서함
  (Producer)                       (Topic Exchange)           (Queue)

  +----------------------+      +------------------+
  | 기사 태그:            |      |                  |     +-------------------+
  | "sports.soccer.korea"| ---> |  구독 패턴에     | --> | 축구팬 사서함      |
  +----------------------+      |  따라 분류       |     | (sports.soccer.*) |
                                |                  |     +-------------------+
                                |                  |
                                |                  |     +-------------------+
                                |                  | --> | 스포츠 종합 사서함 |
                                |                  |     | (sports.#)        |
                                +------------------+     +-------------------+
```

### 패턴 매칭 규칙: `*`와 `#`의 차이

routing_key는 `.`(점)으로 단어를 구분합니다. 예: `sports.soccer.korea` (3개의 단어)

| 와일드카드 | 의미 | 신문 구독 비유 |
|-----------|------|--------------|
| `*` (별표) | **정확히 1개** 단어와 매칭 | "스포츠의 어떤 종목이든, 한국 기사만 보겠다" |
| `#` (샵) | **0개 이상**의 단어와 매칭 | "스포츠라면 뭐든지 다 보겠다" |

### 단계별 매칭 예시

기사(메시지)의 routing_key가 `sports.soccer.korea`일 때, 각 구독 패턴이 매칭되는지 확인해봅시다:

```text
기사 태그(routing_key): sports.soccer.korea
                        [단어1].[단어2].[단어3]

구독 패턴(binding_key)          매칭 여부     이유
---------------------------------------------------------------------------
sports.soccer.korea             [O] 매칭     정확히 일치
sports.soccer.*                 [O] 매칭     * = "korea" (1개 단어 매칭)
sports.*.korea                  [O] 매칭     * = "soccer" (1개 단어 매칭)
sports.#                        [O] 매칭     # = "soccer.korea" (2개 단어 매칭)
#                               [O] 매칭     # = 전체 (모든 메시지 수신)
sports.soccer                   [X] 불일치   단어가 2개뿐 (3개 필요)
sports.baseball.*               [X] 불일치   baseball != soccer
*.soccer                        [X] 불일치   단어가 2개뿐 (3개 필요, *는 1개만 매칭)
```

### 실전 시나리오

```text
[주문 이벤트 라우팅 예시]

  routing_key 구조:  order.{이벤트}.{국가}

  발행되는 메시지들:
    order.created.kr     (한국 주문 생성)
    order.created.us     (미국 주문 생성)
    order.cancelled      (주문 취소 -- 국가 없음, 단어 2개)

  구독 패턴별 수신 결과:
  +---------------------+------------------------------------------+
  | binding_key         | 수신하는 메시지                            |
  +---------------------+------------------------------------------+
  | order.created.*     | order.created.kr, order.created.us       |
  |                     | (생성 이벤트만, 모든 국가)                  |
  +---------------------+------------------------------------------+
  | order.*.kr          | order.created.kr                         |
  |                     | (한국 주문만, 모든 이벤트)                  |
  +---------------------+------------------------------------------+
  | order.#             | order.created.kr, order.created.us,      |
  |                     | order.cancelled  (모든 주문 이벤트)        |
  +---------------------+------------------------------------------+
```

> **Direct vs Topic**: Direct는 routing_key가 **정확히** 일치해야 하지만, Topic은 **패턴**으로 유연하게 매칭합니다.
> 라우팅 조건이 복잡해질수록 Topic Exchange가 유리합니다.

Topic Exchange에 서로 다른 패턴의 binding key로 큐를 바인딩합니다.

In [ ]:
# 특정 패턴 바인딩: order.created.* (생성 이벤트만)
resp = await client.post(
    "/rabbitmq/topic/bind",
    params={"queue_name": "order-created-q", "binding_key": "order.created.*"}
)
print(f"Bind created: {resp.json()}")

In [ ]:
# 와일드카드 바인딩: order.# (모든 주문 이벤트)
resp = await client.post(
    "/rabbitmq/topic/bind",
    params={"queue_name": "all-orders-q", "binding_key": "order.#"}
)
print(f"Bind all: {resp.json()}")

`order.created.kr` routing key로 메시지를 발행합니다. 두 큐 모두 매칭되어야 합니다.

In [ ]:
resp = await client.post(
    "/rabbitmq/topic/publish",
    json={"routing_key": "order.created.kr", "content": "new order from KR"}
)
print(f"Topic publish: {resp.json()}")

`order.cancelled` routing key로 발행하면 `order.#`만 매칭되고, `order.created.*`는 매칭되지 않습니다.

In [ ]:
resp = await client.post(
    "/rabbitmq/topic/publish",
    json={"routing_key": "order.cancelled", "content": "order cancelled"}
)
print(f"Topic publish (cancelled): {resp.json()}")

큐 상태를 비교합니다. `order-created-q`는 1개, `all-orders-q`는 2개의 메시지가 있어야 합니다.

In [ ]:
for q in ["order-created-q", "all-orders-q"]:
    resp = await client.get(f"/rabbitmq/queue/info/{q}")
    print(f"{q}: {resp.json()}")

---
## 7. Pattern 4: DLQ (Dead Letter Queue)

### 병원 응급실로 이해하기

DLQ는 **병원의 중환자실 이송 시스템**과 같습니다.

일반 진료실(Main Queue)에서 의사(Consumer)가 환자(메시지)를 치료합니다.
치료에 성공하면 퇴원(ACK)시키고, 치료에 실패하면 중환자실(DLQ)로 이송합니다.
중환자실로 이송하는 규칙을 DLX(Dead Letter Exchange)라고 합니다.

```text
[병원 응급실 비유]

  환자 접수              일반 진료실                의사
  (Producer)           (Main Queue)            (Consumer)
                                                  |
  +---------+      +------------------+      +----+----+
  | 환자    | ---> |  payment-queue   | ---> | 치료    |
  | 접수    |      |  (x-dead-letter- |      | 시도    |
  +---------+      |  exchange 설정됨)|      +----+----+
                   +------------------+           |
                                             성공? |  실패?
                                            +-----+------+
                                            |            |
                                         [ACK]        [NACK]
                                         퇴원!      (requeue=False)
                                                        |
                                                        v
                   +------------------+      +------------------+
                   |  payment-queue   | <--- | payment-queue    |
                   |  .dlq            |      | .dlx (DLX)      |
                   |  (중환자실)       |      | (이송 규칙)      |
                   +------------------+      +------------------+
                          |
                          v
                   관리자가 확인 후
                   재처리 or 폐기
```

### 왜 DLQ가 필요한가?

메시지 처리가 실패하는 경우를 생각해봅시다:
- 결제 API가 일시적으로 다운됨
- 메시지 데이터가 잘못되어 파싱 불가
- 비즈니스 로직 예외 발생

이런 실패 메시지를 그냥 버리면 **데이터가 유실**되고,
계속 재시도하면 **같은 에러가 무한 반복**됩니다.

DLQ는 이 문제를 해결합니다: **실패한 메시지를 별도의 큐에 격리**해서,
나중에 관리자가 원인을 파악하고 재처리할 수 있게 합니다.

### 메시지 처리 흐름도

```text
메시지 도착
    |
    v
+-------------------+
| Consumer가 처리    |
| 시도               |
+---+----------+----+
    |          |
    v          v
 [성공]      [실패]
    |          |
    v          v
 ACK 전송    NACK 전송 (requeue=False)
    |          |
    v          v
 큐에서      DLX를 통해
 삭제됨      DLQ로 이동
               |
               v
         +------------+
         | DLQ에 보관  |
         +------+-----+
                |
          +-----+------+
          |            |
          v            v
     관리자가       원인 파악 후
     모니터링      메인 큐로 재발행
```

### DLQ 구성 요소 정리

| 구성 요소 | 병원 비유 | RabbitMQ에서의 역할 |
|----------|---------|-------------------|
| Main Queue | 일반 진료실 | 메시지가 처음 도착하는 큐. `x-dead-letter-exchange` 인자로 DLX를 지정함 |
| Consumer | 의사 | 메시지를 처리하고, 성공 시 ACK / 실패 시 NACK |
| DLX | 중환자실 이송 규칙 | Fanout 타입 Exchange. 실패한 메시지를 DLQ로 라우팅 |
| DLQ | 중환자실 | 실패한 메시지가 격리되는 큐. `{queue}.dlq` 이름으로 생성 |
| x-death 헤더 | 환자 차트(병력) | 실패 원인, 원래 큐, 실패 횟수 등 추적 정보 포함 |

DLQ 세트를 구성합니다. 메인 큐와 DLQ/DLX가 함께 생성됩니다.

In [ ]:
resp = await client.post("/rabbitmq/dlq/setup", params={"queue_name": "payment-queue"})
print(f"DLQ setup: {resp.json()}")

메인 큐에 메시지를 발행합니다. Consumer가 nack하면 DLQ로 이동하게 됩니다.

In [ ]:
resp = await client.post(
    "/rabbitmq/direct/publish",
    params={"queue_name": "payment-queue"},
    json={"content": "payment", "metadata": {"payment_id": "PAY-001", "amount": 50000}}
)
print(f"Publish to main queue: {resp.json()}")

DLQ에 쌓인 메시지를 조회합니다. Consumer가 처리를 실패한 메시지가 여기에 격리됩니다.

In [ ]:
resp = await client.get("/rabbitmq/dlq/messages", params={"queue_name": "payment-queue"})
print(f"DLQ messages: {resp.json()}")

### DLQ 활용 팁

- DLQ 메시지에는 `x-death` 헤더가 포함되어 실패 원인을 추적할 수 있습니다
- 주기적으로 DLQ를 모니터링하여 시스템 문제를 조기에 발견하세요
- 재처리가 가능한 메시지는 DLQ에서 꺼내 메인 큐로 다시 발행할 수 있습니다

### Manual ACK/NACK 패턴

RabbitMQ의 메시지 처리 보장은 **ACK/NACK** 메커니즘에 기반합니다.

| 동작 | 메서드 | 효과 |
|------|--------|------|
| **ACK** | `message.ack()` | 처리 완료 → 큐에서 삭제 |
| **NACK** | `message.nack(requeue=True)` | 처리 실패 → 큐로 복귀 (재시도) |
| **NACK** | `message.nack(requeue=False)` | 처리 실패 → DLQ로 이동 |
| **Reject** | `message.reject(requeue=False)` | 단일 메시지 거부 → DLQ |

우리 코드에서 `async with message.process():`는 내부적으로:
- 정상 종료 시 → **자동 ACK**
- 예외 발생 시 → **자동 NACK (requeue=True)**

```python
# 프로젝트 코드 (rabbitmq_broker.py:92-95)
async with queue.iterator() as queue_iter:
    async for message in queue_iter:
        async with message.process():  # ← 자동 ACK/NACK
            data = json.loads(message.body.decode())
            await callback(data)
```

**수동 NACK가 필요한 경우**: 비즈니스 로직에 따라 재시도 vs DLQ 결정이 필요할 때
→ `message.process()` 대신 직접 `ack()`/`nack()` 호출

---
## 8. Pattern 5: Priority Queue

### 병원 응급 분류(Triage)로 이해하기

Priority Queue는 **병원 응급실의 환자 분류 시스템(Triage)**과 같습니다.

응급실에 환자가 도착하면, 증상의 심각도에 따라 **우선순위**가 매겨집니다.
대기실(Queue)에 먼저 온 환자라도, 뒤에 더 심각한 환자가 오면 **심각한 환자가 먼저 치료**됩니다.

```text
[병원 응급 분류(Triage) 비유]

  환자 도착 순서:  1번(감기) -> 2번(심정지) -> 3번(골절)

  대기실 (Priority Queue, x-max-priority=10):
  +----------------------------------------------------------+
  |                                                          |
  |  [priority=10] 2번 환자: 심정지  --> 즉시 치료!            |
  |  [priority= 5] 3번 환자: 골절    --> 다음 치료            |
  |  [priority= 0] 1번 환자: 감기    --> 대기                 |
  |                                                          |
  |  * 먼저 왔어도(1번), 우선순위가 낮으면 뒤로 밀림             |
  |  * 같은 우선순위면 먼저 온 순서대로 (FIFO)                  |
  +----------------------------------------------------------+
        |
        v
  의사(Consumer)는 우선순위 높은 환자부터 치료:
  2번(심정지) --> 3번(골절) --> 1번(감기)
```

### 핵심 개념

**같은 큐에 있어도 우선순위가 높은 메시지가 먼저 소비됩니다.**

일반 큐는 FIFO(First In, First Out) -- 먼저 들어온 메시지가 먼저 나갑니다.
Priority Queue는 이 규칙을 깨고, **우선순위가 높은 메시지를 먼저 꺼냅니다.**

| 우선순위 | 비유 | 예시 |
|---------|------|------|
| priority=10 | 심정지 환자 (즉시 치료) | 시스템 장애 알림, 긴급 결제 취소 |
| priority=5 | 골절 환자 (빠른 치료) | 일반 주문 처리, 중요 알림 |
| priority=0 | 감기 환자 (대기) | 로그 수집, 통계 집계, 비긴급 작업 |

### 설정 방법

- **큐 생성 시**: `x-max-priority=10`으로 최대 우선순위 범위를 지정 (0~10)
- **메시지 발행 시**: `priority=N`으로 각 메시지의 우선순위를 지정
- 이 값은 **큐 생성 시 한 번만** 설정할 수 있습니다 (이후 변경 불가)

### [주의] prefetch_count와의 상호작용

Priority Queue를 사용할 때 **반드시 알아야 할 함정**이 있습니다:

```text
[prefetch_count가 크면 우선순위가 무시될 수 있음]

  상황: prefetch_count=10, Consumer 1대

  1) 낮은 우선순위 메시지 10개가 먼저 도착
     --> Consumer가 10개를 미리 가져감 (prefetch)

  2) 높은 우선순위 메시지가 뒤늦게 도착
     --> Consumer가 이미 10개를 들고 있으므로, 큐에서 대기

  3) 결과: 높은 우선순위 메시지가 나중에 처리됨!

  해결: prefetch_count=1로 설정하면 우선순위가 정확히 동작
        (단, 처리량이 떨어질 수 있으므로 트레이드오프 고려)
```

> **정리**: 우선순위를 정확하게 지키려면 `prefetch_count`를 낮게 설정하세요.
> 이미 Consumer에게 전달(prefetch)된 메시지는 우선순위 재정렬이 되지 않습니다.

서로 다른 우선순위로 메시지 3개를 발행합니다. 소비 시 우선순위가 높은 순서대로 처리됩니다.

In [ ]:
priorities = [
    (1, "low-priority task"),
    (10, "CRITICAL task"),
    (5, "medium-priority task"),
]
for prio, desc in priorities:
    resp = await client.post(
        "/rabbitmq/priority/publish",
        params={"queue_name": "task-queue"},
        json={"content": desc, "priority": prio}
    )
    print(f"Priority {prio:>2}: {resp.json()}")

큐 상태를 조회하여 메시지가 쌓였는지 확인합니다.

In [ ]:
resp = await client.get("/rabbitmq/queue/info/task-queue")
print(f"Task queue info: {resp.json()}")

### Priority Queue 소비 순서

Consumer가 큐에서 메시지를 가져올 때, RabbitMQ는 내부적으로 우선순위별 서브큐를 유지합니다.  
위 예제에서 소비 순서는: `CRITICAL(10)` → `medium(5)` → `low(1)` 이 됩니다.

> **참고**: 메시지가 이미 Consumer에게 전달(prefetch)된 상태라면 우선순위 재정렬이 발생하지 않습니다.

---
## 9. Pattern 6: TTL (Time-To-Live)

### 음식 유통기한으로 이해하기

TTL은 **음식의 유통기한**과 같습니다.

마트에서 파는 음식에는 유통기한이 있습니다.
유통기한이 지나면 **자동으로 폐기**됩니다.
RabbitMQ 메시지도 마찬가지로, TTL을 설정하면 지정 시간이 지난 메시지가 자동으로 삭제됩니다.

```text
[음식 유통기한 비유]

  냉장고 (Queue):
  +----------------------------------------------------------+
  |                                                          |
  |  [TTL 없음]     통조림       --> 유통기한 없음, 영원히 보관  |
  |  [TTL=30초]    신선한 우유   --> 30초 후 자동 폐기          |
  |  [TTL=5초]     회(생선)      --> 5초 후 자동 폐기           |
  |                                                          |
  |  * 유통기한이 지나면 냉장고에서 자동으로 사라짐               |
  |  * 누군가(Consumer)가 기한 내에 꺼내가면 정상 소비          |
  +----------------------------------------------------------+
```

### 두 가지 TTL 설정 방식

TTL을 설정하는 방법은 두 가지가 있습니다:

| 방식 | 비유 | 설정 방법 | 특징 |
|------|------|----------|------|
| **메시지 단위 TTL** | 각 음식마다 다른 유통기한 라벨 | `Message(expiration="5000")` | 메시지마다 다른 TTL 가능 |
| **큐 단위 TTL** | 냉장고 전체에 "3일 후 전부 폐기" 규칙 | `x-message-ttl=5000` (큐 생성 시) | 큐의 모든 메시지에 동일 적용 |

```text
[메시지 단위 TTL]                    [큐 단위 TTL]

Queue:                              Queue (x-message-ttl=10000):
+------------------+                +------------------+
| msg A: TTL=5초   |                | msg A: 10초 후   |
| msg B: TTL=30초  |                | msg B: 10초 후   |
| msg C: TTL 없음  |                | msg C: 10초 후   |
+------------------+                +------------------+

각 메시지가 서로 다른                모든 메시지가 동일한
시간에 만료됨                       시간에 만료됨
```

우리 프로젝트에서는 **메시지 단위 TTL**을 사용합니다.

### TTL + DLQ 조합: "유통기한 지난 식품을 폐기 목록에 기록"

TTL이 만료된 메시지를 그냥 삭제하면 추적이 불가능합니다.
하지만 **DLQ와 조합**하면, 만료된 메시지를 DLQ에 보관하여 나중에 확인할 수 있습니다.

이것은 마트에서 **유통기한 지난 식품을 그냥 버리지 않고, 폐기 목록에 기록**하는 것과 같습니다.

```text
[TTL + DLQ 조합]

  메시지 발행 (TTL=5초)
       |
       v
  +------------------+        5초 경과, 아무도 안 가져감
  |  Main Queue      | -----> TTL 만료!
  |  (x-dead-letter- |              |
  |  exchange 설정됨) |              v
  +------------------+     +------------------+     +------------------+
                           |  DLX (Exchange)  | --> |  DLQ             |
                           +------------------+     |  (만료 메시지     |
                                                    |   보관소)        |
                                                    +------------------+
                                                           |
                                                           v
                                                    관리자가 "왜 이 메시지가
                                                    시간 내에 처리되지 못했는지"
                                                    원인 분석 가능
```

> **실전 활용**: 주문 결제 대기 시간(예: 30분)을 TTL로 설정하고,
> 시간 내에 결제가 완료되지 않으면 DLQ로 보내 자동 주문 취소 처리를 할 수 있습니다.

TTL이 짧은(5초) 메시지를 발행합니다. 5초 후에 큐에서 자동 삭제됩니다.

In [ ]:
resp = await client.post(
    "/rabbitmq/ttl/publish",
    params={"queue_name": "temp-queue"},
    json={"content": "expires in 5 seconds", "ttl_ms": 5000}
)
print(f"TTL publish (5s): {resp.json()}")

발행 직후 큐 상태를 확인하면 메시지가 존재합니다.

In [ ]:
resp = await client.get("/rabbitmq/queue/info/temp-queue")
print(f"즉시 조회: {resp.json()}")

6초 후에 다시 조회하면 TTL이 만료되어 메시지가 사라진 것을 확인할 수 있습니다.

In [ ]:
import asyncio

print("6초 대기 중...")
await asyncio.sleep(6)

resp = await client.get("/rabbitmq/queue/info/temp-queue")
print(f"6초 후 조회: {resp.json()}")

### TTL + DLQ 조합

TTL이 만료된 메시지를 DLQ로 보내도록 설정하면, 만료된 메시지도 추적할 수 있습니다.  
이는 `x-dead-letter-exchange`와 `expiration`을 함께 사용하여 구현합니다.

```text
[만료된 메시지] ──▶ DLX ──▶ DLQ (만료 메시지 보관)
```


---
## 10. RabbitMQ Management UI 활용법

RabbitMQ는 웹 기반 관리 UI를 제공합니다.

| 항목 | 값 |
|------|----|
| URL | http://localhost:15672 |
| Username | `guest` |
| Password | `guest` |

### 주요 메뉴

```text
┌─────────────────────────────────────────────────────────┐
│  RabbitMQ Management                                    │
├──────────┬──────────┬──────────┬──────────┬────────────┤
│ Overview │ Connect- │ Channels │ Exchanges│   Queues   │
│          │ ions     │          │          │            │
├──────────┴──────────┴──────────┴──────────┴────────────┤
│                                                         │
│  [Overview]   - 전체 메시지 비율, 노드 상태              │
│  [Queues]     - 큐별 메시지 수, 소비자 수, 메시지 비율   │
│  [Exchanges]  - Exchange 목록, 타입, 바인딩 정보         │
│  [Connections]- 연결된 클라이언트 목록                    │
│  [Channels]   - 채널별 prefetch, unacked 수             │
│                                                         │
└─────────────────────────────────────────────────────────┘
```

### 실습에서 확인할 것들

1. **Queues** 탭에서 `order-queue`, `task-queue`, `payment-queue.dlq` 등 확인
2. **Exchanges** 탭에서 `events-exchange`(fanout), `order-events`(topic) 확인
3. 각 큐를 클릭하면 **Get Messages** 버튼으로 메시지 내용을 직접 확인 가능

---
## 11. 정리 및 요약

### 6가지 패턴 비교 테이블

| 패턴 | Exchange 타입 | Routing Key | 사용 사례 | 메서드 |
|------|-------------|-------------|-----------|--------|
| **Direct Queue** | Default (`""`) | 큐 이름 | 1:1 작업 분배 | `publish()` |
| **Fanout** | FANOUT | 무시됨 | 브로드캐스트, 알림 | `publish_fanout()` |
| **Topic** | TOPIC | 패턴 (`*`, `#`) | 선택적 라우팅 | `publish_topic()` |
| **DLQ** | FANOUT (DLX) | - | 실패 메시지 격리 | `setup_dlq()` |
| **Priority** | Default (`""`) | 큐 이름 | 긴급 작업 우선 처리 | `publish_priority()` |
| **TTL** | Default (`""`) | 큐 이름 | 임시 데이터, 캐시 | `publish_ttl()` |

### API 엔드포인트 요약

| 메서드 | 엔드포인트 | 패턴 |
|--------|-----------|------|
| POST | `/rabbitmq/direct/publish` | Direct |
| POST | `/rabbitmq/fanout/publish` | Fanout |
| POST | `/rabbitmq/fanout/bind` | Fanout 바인딩 |
| POST | `/rabbitmq/topic/publish` | Topic |
| POST | `/rabbitmq/topic/bind` | Topic 바인딩 |
| POST | `/rabbitmq/dlq/setup` | DLQ 구성 |
| GET  | `/rabbitmq/dlq/messages` | DLQ 조회 |
| POST | `/rabbitmq/priority/publish` | Priority |
| POST | `/rabbitmq/ttl/publish` | TTL |
| GET  | `/rabbitmq/queue/info/{queue}` | 큐 정보 |

### 핵심 설계 원칙

1. **내구성(Durability)**: `durable=True` + `DeliveryMode.PERSISTENT`로 메시지 유실 방지
2. **흐름 제어(Flow Control)**: `prefetch_count=10`으로 Consumer 과부하 방지
3. **자동 복구**: `connect_robust()`로 네트워크 장애 시 자동 재연결
4. **실패 격리**: DLQ 패턴으로 실패한 메시지를 별도 관리

### RabbitMQ 4 — 무엇이 달라졌나?

이 프로젝트는 **RabbitMQ 4**를 사용합니다. 3.x 대비 주요 변경사항:

| 변경사항 | 설명 |
|----------|------|
| **Quorum Queue 기본화** | Raft 합의 기반 복제 큐가 표준. Classic Mirrored Queue 제거 |
| **Classic Queue 유지** | 비복제 Classic Queue는 여전히 사용 가능 (이 프로젝트에서 사용하는 방식) |
| **기본 재전송 제한** | Quorum Queue의 기본 delivery limit = 20 (무한 재시도 방지) |
| **AMQP 1.0 네이티브** | AMQP 1.0 프로토콜 직접 지원 (0-9-1도 계속 지원) |

```text
[큐 타입 선택 가이드 - RabbitMQ 4]

  데이터 안전이 중요한가? ──→ Yes ──→ Quorum Queue (Raft 복제)
           │                              • 3+ 노드 클러스터에서 사용
           │                              • 기본 delivery limit = 20
           │
           └──→ No ──→ Classic Queue (단일 노드)
                         • 개발/학습 환경에 적합
                         • 이 프로젝트에서 사용하는 방식
```

> **왜 이 프로젝트에서는 Classic Queue를 사용하는가?**
> 단일 노드 학습 환경에서는 Quorum Queue의 복제 이점이 없습니다.
> 프로덕션에서 RabbitMQ 클러스터를 운영한다면 Quorum Queue를 기본으로 사용하세요.

마지막으로 httpx 클라이언트를 정리합니다.

In [ ]:
await client.aclose()
print("httpx client closed.")